# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

To explore the metadata structure, let's list all available record sets, their `@id`s, and their fields (also referenced by `@id`).

In [ ]:
record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else ''}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, column: {field.column})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll extract data from all record sets. Each will be loaded into a pandas DataFrame for further analysis.

In [ ]:
# Compile the list of record set @id's
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded {len(records)} records from RecordSet {record_set_id}")
    if len(records) > 0:
        print(f"Sample columns for {record_set_id}: {dataframes[record_set_id].columns.tolist()}")

# For demonstration, take the first available record set as primary for EDA
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nSample data from {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's:
- Select a numeric field (e.g., `cr:field/protein_abundance` if available),
- Filter based on a threshold,
- Normalize values,
- Optionally group by another field for aggregation.

All field, record set, or column names are referenced by their `@id` as required.

In [ ]:
# Identify a numeric field in the main DataFrame
df = dataframes[main_record_set_id]
numeric_field_candidates = df.select_dtypes(include=['number']).columns.tolist()
print(f"Numeric fields found: {numeric_field_candidates}")

if numeric_field_candidates:
    numeric_field = numeric_field_candidates[0]  # Take the first numeric field by @id
    threshold = df[numeric_field].mean()  # Use mean as initial threshold
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the selected numeric field
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Attempt to group by first available object/categorical field by @id
    group_field_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
    print(f"Group-by candidates: {group_field_candidates}")
    if group_field_candidates:
        group_field = group_field_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped mean {numeric_field} by {group_field}:")
        display(grouped_df.head())
else:
    print("No numeric field found for EDA in this record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll show a histogram or bar chart of the selected numeric field and, if grouping is possible, a grouped bar chart.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_candidates:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field} in {main_record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()

    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(10, 5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field)
        plt.title(f"Mean of {numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(f"mean({numeric_field})")
        plt.xticks(rotation=90)
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

*In this notebook, we successfully loaded and explored the FAIR^2 dataset on mass spectrometry analysis of extracellular vesicles using `mlcroissant`, identified record sets and fields by `@id`, and performed basic EDA and visualization steps. For further insights, perform domain-specific analyses or modeling by referencing field and column `@id`s as explored above.*